In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# step1:-problem defining
# the problem is that predicting the loan_approval -yes or not

In [ ]:
# step2:-aceesing the data from file to dataframe for performing the operation on it
df=pd.read_csv("loan_approval_data.csv")
df.head()

In [ ]:
# step3:-analysing and exploring the data for meaningful insights
df.info()

In [ ]:
df.isnull().sum() # return how many null are present in each col

In [ ]:
# now we have an 50 null value to each col but we cant delete it its bad cleaning but we can fill that null
# so in our data set there is categorical data and numerical data so numerical-mean fill and cat-mode
# so we accces the data seprately
# # numerical
num_col=df.select_dtypes(include=['number']).columns
# categorical
cat_col=df.select_dtypes(include=['object']).columns

In [ ]:
# to clean and filling the data in ml we use sklearn imputer
from sklearn.impute import SimpleImputer
simp=SimpleImputer(strategy='mean')
df[num_col]=simp.fit_transform(df[num_col])

# cat
simp=SimpleImputer(strategy='most_frequent')
df[cat_col]=simp.fit_transform(df[cat_col])

In [ ]:
df.isnull().sum()

In [ ]:
# now our data is cleaned and perfect
# now we visualize our data and seen relationship and calculate meaningful insight
# EDA-exploratory data analysis
# Visualization
import seaborn as sns
import matplotlib.pyplot as plt

# now we seen the all visualization first we seen our label that is loan approval how many true and false
c=df['Loan_Approved'].value_counts()
plt.pie(c,labels=['No','Yes'],autopct='%1.1f%%')

In [ ]:
# now we can seen married and loan approved
sns.barplot(
    data=df,
    x='Credit_Score',
    y='Loan_Approved',
    hue='Marital_Status'
)


In [ ]:
ax=sns.barplot(
    data=df,   
    y='Applicant_Income',
    x='Employer_Category',
    hue='Loan_Approved'
    
)
ax.bar_label(ax.containers[0])


In [ ]:
ax=sns.barplot(
    data=df,   
    y='Credit_Score',
    x='Employer_Category',
    hue='Loan_Approved'
    
)
ax.bar_label(ax.containers[0])

In [ ]:
# now we seen for outlier using box plot
a,b=plt.subplots(2,2)
sns.boxplot(ax=b[0,0],data=df,x='Loan_Approved',y='Applicant_Income')

In [ ]:
sns.histplot(
    data=df,
    x='Credit_Score',
    hue='Loan_Approved',
    bins=10
    
)

In [ ]:
# Encoding
# convewrsion of each categories into sepreate columns
# In sklearn for encoding we having two type
# 1.LabelEncoder: its neccesary to first use
# its for columns that are label and that columns you encode in same col not seprate columns
# 2.OneHotEncoder:
# its convert the each categories into seprate columns

# we import both two

from sklearn.preprocessing import OneHotEncoder,LabelEncoder

le=LabelEncoder()
df['Loan_Approved']=le.fit_transform(df['Loan_Approved'])
df['Education_Level']=le.fit_transform(df['Education_Level'])
# 2.OneHotEncoder
# to encode the data we need categorical data from dataframe
# cat_col=df.select_dtypes(include="object").columns

# now we have all columns but we dont need two columns that we use in labelencoder so remove that

# drop_cat_col=df[cat_col].drop(columns=['Loan_Approved','Education_Level']).columns

drop_cat_col=['Employment_Status','Marital_Status','Loan_Purpose','Property_Area','Gender','Employer_Category']

# now we use OneHotEncoder
ohe=OneHotEncoder(drop='first',sparse_output=False,handle_unknown='ignore')

# now we fit our data that we have to one hot
encode=ohe.fit_transform(df[drop_cat_col]) #it not return dataframe it return 2d so convert to df

# but dataframe require new columns that had been one hot codeded so for this
new_col=ohe.get_feature_names_out(drop_cat_col)

# now we can convert our data to dataframe
new=pd.DataFrame(encode,columns=new_col,index=df.index)

df=pd.concat([df.drop(columns=drop_cat_col),new],axis=1)
df.info()

In [ ]:
# now we seen correlation heatmap
df.corr()


In [ ]:
df.corr()["Loan_Approved"]

In [ ]:
plt.figure(figsize=(25,25))
sns.heatmap(
    data=df.corr(),
    annot=True
)
plt.tight_layout()

In [ ]:
# Feature Engineering
# here is one columns that dont affects on the result i.e id
# so we remove it
df=df.drop('Applicant_ID',axis=1)

# we break our df into feature and label
x=df.drop('Loan_Approved',axis=1)
y=df['Loan_Approved']

In [ ]:
# now one this is main we scale the all data but before that we split it into 80:20
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

# now we scaling
from sklearn.preprocessing import StandardScaler
std=StandardScaler()
new_x_train=std.fit_transform(x_train)
new_x_test=std.fit_transform(x_test)

In [ ]:
# Training
# we compare eache model and use each model
# KNN
from sklearn.neighbors import KNeighborsClassifier
knc=KNeighborsClassifier(n_neighbors=3)
knc.fit(new_x_train,y_train)
pre1=knc.predict(new_x_test)


# in this project the problem is there is mistake in prediction manually good-bad ....bad-good
# so we required precision and recall
# now we evaluate knc model
from sklearn.metrics import accuracy_score,precision_score,recall_score
print("accuracy knc:",accuracy_score(y_test,pre1))
print("precision_score knc:",precision_score(y_test,pre1))
print("recall_score knc:",recall_score(y_test,pre1))


In [ ]:
# the above evaluation is very weak means model is very weak bcz it require lot of calculation bad model 

In [ ]:
# LogisticRegrresion
from sklearn.linear_model import LogisticRegression
lgr=LogisticRegression()
lgr.fit(new_x_train,y_train)
pre2=lgr.predict(new_x_test)

from sklearn.metrics import accuracy_score,precision_score,recall_score
print("accuracy lgr:",accuracy_score(y_test,pre2))
print("precision_score lgr:",precision_score(y_test,pre2))
print("recall_score lgr:",recall_score(y_test,pre2))

In [ ]:
#naive baye's
from sklearn.naive_bayes import GaussianNB
nb=GaussianNB()
nb.fit(new_x_train,y_train)
pre3=nb.predict(new_x_test)

from sklearn.metrics import accuracy_score,precision_score,recall_score
print("accuracy nb:",accuracy_score(y_test,pre3))
print("precision_score nb:",precision_score(y_test,pre3))
print("recall_score nb:",recall_score(y_test,pre3))

In [ ]:
# till now the LogisticRegression is best for this but the truth is that it is depend on data and the model
# result vary